# Generative large language models

In this labs we will look at the current state of the art in natural language processing - generative large language models based on the transformer architecture.

## Using the models

Install the required libraries. We'll mostly be using huggingface. Accelerate is used to train models on a GPU while torch and torchvision are two additional libraries for training and running neural networks.

In [1]:
!pip install torch torchvision
!pip install accelerate
!pip install -U huggingface_hub hf_xet
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 4.9 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 11.0 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
# Some versions of huggingface hang when downloading models.
# Run this if your models stop downloading at a certain point.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Qwen language model with 0.6B parameters
model_name = "Qwen/Qwen3-0.6B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto" # Device used to run the model (e.g. CPU/GPU)
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

W1122 15:32:18.492000 411 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 15:32:18.530000 411 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Huggingface LLMs take input as a sequence of messages, where every message contains a role and the content.

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short joke about saving RAM."},
]

Every model has a different way of formatting and tokenizing text. apply_chat_template correctly prepares text for the selected model.

In [5]:
# Process input
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)



In [6]:
# Generate outputs
outputs = model.generate(**inputs, max_new_tokens=1024)

# Remove formatting added by apply_chat_template
clean_text = tokenizer.decode(outputs[0])

print(clean_text)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a short joke about saving RAM.<|im_end|>
<|im_start|>assistant
<think>
Okay, the user wants a short joke about saving RAM. Let me think about how to approach this.

First, I need to make sure the joke is clear and easy to understand. RAM is a computer component, so saving it would relate to managing resources. Maybe play on the idea of using RAM efficiently.

I should start with a relatable scenario. Like, someone who uses a lot of memory. Then, the punchline could be about not wasting resources. Maybe something like saving a memory card or a computer's RAM by using it smartly.

Let me check if there's a common pun or wordplay. "RAM" is a common abbreviation, so that's a good point. The punchline could be about not overusing it. Maybe "saving RAM means not overusing it, like saving a memory card."

Wait, the user said "short," so I need to keep it concise. Maybe use a simple structure with a setup and a pu

In [7]:
# We can disable thinking in the chat template
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
 enable_thinking=False
).to(model.device)

# We can remove the input part of the result
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=1024)

# Remove formatting added by apply_chat_template
clean_text = tokenizer.decode(outputs[0][input_len:])

print(clean_text)

Oh, right, saving RAM is like saving your phone battery—when you save a joke, your phone gets a little more power!<|im_end|>


A similar approach would work with other Huggingface models

## Finetuning models

The models are capable as-is, but we can further fine-tune them to either improve performance or to give them information related to a specific tasks. Some popular fine-tuning methods include:

* Supervised fine-tuning (SFT)
* Low rank adalters (LoRA)
* Reinforcement learning with human feedback (RLHF)
* Direct preference optimization (DPO)

## Supervised fine-tuning

Trains a model to on a pair of input and output texts. This is useful when we want to specialize a model for a specific task (for example, text classification) and have annotated data related to that task.

Let's attempt to fine-tune this model for sentiment classification of movie reviews.

In [8]:
from datasets import load_dataset

# Download the IMDb dataset
imdb_dataset = load_dataset('stanfordnlp/imdb')

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [9]:
print(imdb_dataset["train"]["text"])

Column(['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far bet

Let's see how the model performs without fine-tuning

In [10]:
# Generate messages in the same format as before
def create_messages(texts):
    role = "You are a helpful assistant trying to determine the sentiment of a movie review."
    user = "Does the following review have a positive or negative sentiment. Answer only with <positive> or <negative> and nothing else."
    review = texts
    messages = [
        {"role": "system", "content": role},
        {"role": "user", "content": user + "\n" + review},
    ]
    return messages

In [11]:
from sklearn.utils import shuffle

# Let's see what the model returns on the first 50 reviews
reviews, labels = shuffle(imdb_dataset["test"]["text"], imdb_dataset["test"]["label"])
for i in range(50):
    review = reviews[i]
    label = labels[i]
    messages_to_test = create_messages(review)

    inputs = tokenizer.apply_chat_template(
    	messages_to_test,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
      enable_thinking=False
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    # Generate outputs
    outputs = model.generate(**inputs, max_new_tokens=256)

    # Remove formatting added by apply_chat_template
    clean_text = tokenizer.decode(outputs[0][input_len:])
    #print(messages_to_test)
    print(clean_text, label)

<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><

It seems to already work quite well - it's a common task that LLMs should generally be quite good at.

Question: Can we guarantee the model wasn't already trained on this data?

## Create fine-tuning dataset

The dataset needs pairs of inputs and desired outputs. We need to convert out dataset into a suitable format. Each training example will have the user's prompt and the desired response from the model.

We will perform training using the trl library.

In [12]:
def create_training_messages(review, label):
    role = "You are a helpful assistant trying to determine the sentiment of a movie review."
    user = "Does the following review have a positive or negative sentiment. Answer only with <positive> or <negative> and nothing else."
    messages = {"messages":
                [{"role": "system", "content": role},
                 {"role": "user", "content": user + "\n" + review},
                 {"role": "assistant", "content": label}] #+ "<|im_end|><|endoftext|>"}]
               }
    return messages

In [13]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 19.6 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [14]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset


train_reviews, train_labels = shuffle(imdb_dataset["train"]["text"], imdb_dataset["train"]["label"])
train_labels = ["positive" if x else "negative" for x in train_labels]

# TRL requires the inputs to be in the huggingface dataset format
# Dataset.from list converts a list into that format
messages = Dataset.from_list([create_training_messages(x, y)
                              for x, y in zip(train_reviews[:20], train_labels[:20])])

#Specify the training arguments
training_args = SFTConfig(num_train_epochs=1, # Number of epochs
                          assistant_only_loss=True, #Calculate loss only on the outputs
                          fp16=True,)
                          #eos_token="<|im_end|>")
# Initialize the trainer
trainer = SFTTrainer(
    model="Qwen/Qwen3-0.6B",  # Start from a pretrained model.
    train_dataset=messages,
    args=training_args,

)

#Start training
trainer.train()

# Save the model so that it can be used later
trainer.save_model("./sft_model")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
reviews, labels = shuffle(imdb_dataset["test"]["text"], imdb_dataset["test"]["label"])

# We can load the trained model the same way as any pretrained model
trained_model = AutoModelForCausalLM.from_pretrained(
    "./sft_model",
    device_map="auto" # Device used to run the model (e.g. CPU/GPU)
)

# Repeat the generation code
for i in range(50):
    review = reviews[i]
    label = labels[i]
    messages_to_test = create_messages(review)

    inputs = tokenizer.apply_chat_template(
    	messages_to_test,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
     enable_thinking=False
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    # Generate outputs
    outputs = trained_model.generate(**inputs, max_new_tokens=1024, do_sample=False)

    # Remove formatting added by apply_chat_template
    clean_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

    print(clean_text, label)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 1
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 0
<positive><|im_end|> 1
<positive><

It's difficult to expect improvements on such a small training dataset, but with a larger dataset we could expect the model to improve in sentiment analysis.

# Low rank adaptation (LoRA)

By default, SFTTrainer updates weights across the entire model. This can work with smaller models but requires expensive hardware (and a lot of time!) on larger models.

LoRA works by inserting a smaller number of new weights into the model and only these are trained. This makes training with LoRA much faster, memory-efficient, and produces smaller model weights (a few hundred MBs), which are easier to store and share.

This can be done using the peft library.

In [16]:
!pip install peft
!pip install "trl[peft]"

In [17]:
from peft import LoraConfig

peft_config = LoraConfig( r=16,          #LoRA attention matrix dimension
    lora_alpha=16,                       #Alpha parameter for lora scaling
    target_modules=["q_proj", "v_proj", "k_proj"], #Modules where we apply adapters. Names are model dependent
    lora_dropout=0.1, #Lora dropout
    bias="none", #Lora bias
)

# We can now train on a larger dataset
messages = Dataset.from_list([create_training_messages(x, y)
                  for x, y in zip(train_reviews[:50], train_labels[:50])])

# We pass the peft_config to the SFTT trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=messages,
    args=training_args,
    peft_config=peft_config
)
trainer.train()

# save_model will now only save the lora adapters
trainer.save_model("./lora_model")

Tokenizing train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Step,Training Loss


To load the saved model, first load the base model, then the adapters.

In [18]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
model = PeftModel.from_pretrained(base_model, "./lora_model")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The model can then be used in the same way.

In [19]:
reviews, labels = shuffle(imdb_dataset["test"]["text"], imdb_dataset["test"]["label"])

# Repeat the generation code
for i in range(50):
    review = reviews[i]
    label = labels[i]
    messages_to_test = create_messages(review)

    inputs = tokenizer.apply_chat_template(
    	messages_to_test,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
     enable_thinking=False
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    # Generate outputs
    outputs = trained_model.generate(**inputs, max_new_tokens=1024, do_sample=False)

    # Remove formatting added by apply_chat_template
    clean_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

    print(clean_text, label)

<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 1
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 0
<negative><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<negative><|im_end|> 0
<negative><|im_end|> 0
<positive><|im_end|> 1
<positive><|im_end|> 1
<negative><|im_end|> 0
<positive><|im_end|> 0
<positive><|im_end|> 1
<positive><

# Reinforcement learning with human feedback (RLHF)

So far, we've only looked at text classification models. However, this requires a training dataset with input-output pairs that match a specific problem.

With RLHF, we instead train a model on a reward function. This can either be a simple function that evaluates the model's output, or something more complex (another pre-trained model, or a human-assigned score).

The general process is:

* Start with a general pre-trained model. Problem: the model does not take human feedback in account during training.
* Collect human annotated data and use it to train a reward model (separate model that will score the outputs produced by the original model).
* Use the reinforcement learning to improve original language model with respect to the reward model.

In [20]:
# Restart with initial model
model_name = "Qwen/Qwen3-0.6B"

# Load model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Initialize the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto" # Device used to run the model (e.g. CPU/GPU)
)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In this example, we'll use a simple function that rewards models with 10 words.

In [21]:
# Define a reward function

# Simple option: function that rewards reviews closest to 10 words
# The function takes in the provided prompt, the completions generated by the model, and a number
#   of additional arguments.
def reward_fn(prompts, completions, **kwargs):
    rewards = [1/(abs(10-len(c[0]["content"].split())+0.01)) for c in completions]

    return rewards  # simple length-based reward



In [22]:
from trl import GRPOConfig, GRPOTrainer
import torch

# Define parameters for the PPO trainer that will update the model
grpo_config = GRPOConfig(
    num_train_epochs=1,            # grpo
    learning_rate=1.41e-6,   # Learning rate for the optimizer
    chat_template_kwargs={"enable_thinking": False},  # Disables thinking
    fp16=True
)


In [23]:
# Create a training set
# GRPO requires only the prompt. The responses are generated by the model and then
# scored using the reward function
def generate_grpo_messages():
    role = "You are a helpful assistant trying to write movie review."
    user = "Generate a positive movie rewiev"
    messages = {"prompt":
                [{"role": "system", "content": role},
                 {"role": "user", "content": user}]
               }
    return messages

In [24]:
# Dataset will just be the same prompt repeated 5 times
grpo_messages = Dataset.from_list([generate_grpo_messages()] * 5)

In [25]:
# Create the trainer
grpo_trainer = GRPOTrainer(
    model=model,                       # The pre-trained model
    train_dataset=grpo_messages,       # Dataset to fine-tune on
    args=grpo_config,                  # PPO training configuration
    reward_funcs=reward_fn,             # The reward function used for training
    peft_config=peft_config,            # Supports LoRA

)

# Train the model
grpo_trainer.train()
grpo_trainer.save_model("./grpo_model")

Step,Training Loss


In [26]:
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
model = PeftModel.from_pretrained(base_model, "./grpo_model")

messages = [
    {"role": "system", "content": "You are a helpful assistant trying to write movie review."},
    {"role": "user", "content": "Generate a positive movie rewiev."},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
	enable_thinking=False,
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

# Generate outputs
outputs = model.generate(**inputs, max_new_tokens=256)

# Remove formatting added by apply_chat_template
clean_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

print(clean_text)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Absolutely! Here's a positive movie review:

**"A Heartwarming Journey Through Love and Sacrifice"**  
This film is a beautiful testament to the power of love, compassion, and perseverance. The story follows a young woman named Maya, who faces a harrowing situation to save her family from a devastating natural disaster. Through her journey, we see her growth, resilience, and the strength of her community. The performances are truly commendable, with heartfelt dialogue and emotional depth that leaves a lasting impression. The visuals are stunning, and the soundtrack enhances the emotional impact perfectly. This movie is a must-watch for anyone who loves stories that inspire and uplift.<|im_end|>


# Direct preference optimization (DPO)

Direct preference optimization is similar to RLHF, but does not make use of a reward model. Instead, the generative model is trained on pairs of outputs to the same input, where one output was scored higher than the others. Through training, the model learns match the preffered outputs.

TRL requires a specific format for DPO datasets. Let's examine a large existing dataset for general feedback

In [27]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:5]")

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [28]:
dataset[0]

{'chosen': [{'content': 'Use the pygame library to write a version of the classic game Snake, with a unique twist',
   'role': 'user'},
  {'content': "Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:\n\n1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.\n2. Next, we'll create the game's grid, which will be used to represent the game board. We'll need to define the size of the grid and the spaces within it.\n3. After that, we'll create the snake object, which will be used to represent the player's movement. We'll need to define the size of the snake and the speed at which it moves.\n4. We'll also need to create a food object, which will be used to represent the food that the player must collect to score points. We'll need to define the location of the food and the speed at which it moves.\n5. Once we have these objects se

Each entry has a pair of texts (one chosen, one rejected) and the corresponding score for each entry.

Training a model on this dataset would take a long amount of time. Instead, let's create a DPO dataset of movie reviews that will push our model towards writing positive reviews. Each pair of examples will contain a positive review (chosen) and a negative review (rejected).

In [29]:
def create_dpo_dataset(pos_texts, neg_texts):
    all_examples = []
    prompt = {"content":"Write a movie review",
                  "role":"user"
                 }
    for pos_text, neg_text in zip(pos_texts, neg_texts):
        chosen = [prompt,
                  {"content":pos_text,"role":"assistant"}]
        rejected = [prompt,
                    {"content":neg_text,"role":"assistant"}]
        score_chosen = 10.0
        score_rejected = 0.0
        example = {"chosen":chosen,
                   "rejected":rejected,
                   "score_chosen":score_chosen,
                   "score_rejected":score_rejected}
        all_examples.append(example)
    return all_examples

Create the dataset from the imdb_dataset entries.

In [33]:
from collections import Counter
# Separate positive and negative examples
pos_examples = [t for t, l in zip(imdb_dataset["train"]["text"],
                                  imdb_dataset["train"]["label"]) if l == 1]
neg_examples = [t for t, l in zip(imdb_dataset["train"]["text"],
                                  imdb_dataset["train"]["label"]) if l == 0]

# Create the dataset
list_dataset = create_dpo_dataset(pos_examples[:50], neg_examples[:50])

# Transform it from a list to a proper huggingface dataset format
imdb_dpo_dataset = Dataset.from_list(list_dataset)

Train the model on our data

In [38]:
from trl import DPOTrainer, DPOConfig

model_name = "Qwen/Qwen3-0.6B"

# Load model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Initialize the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto" # Device used to run the model (e.g. CPU/GPU)
)

# As with other types of training, we can specify the training parameters
dpo_config = DPOConfig(num_train_epochs=1,
                      fp16=True,
                       )

# Train LoRA adapters
peft_config = LoraConfig( r=16, #LoRA attention matrix dimension
    lora_alpha=16, #Alpha parameter for lora scaling
    target_modules=["q_proj", "v_proj"], #Modules where we apply adapters. Model dependant
    lora_dropout=0.1, #Lora dropout
    bias="none", #Lora bias

)

# DPOTrainer works mostly the same as the other trainers
trainer = DPOTrainer(
    model="Qwen/Qwen3-0.6B",
    #train_dataset=dataset,
    train_dataset=imdb_dpo_dataset,
    peft_config=peft_config,
    args=dpo_config,


)

# Save the trained model
trainer.train()
trainer.save_model("./dpo_model")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Extracting prompt from train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Step,Training Loss


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

In [39]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
model = PeftModel.from_pretrained(base_model, "./dpo_model")

messages = [

    {"role": "user", "content": "Write a movie review"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
	enable_thinking=False,
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

# Generate outputs
outputs = model.generate(**inputs, max_new_tokens=256)

# Remove formatting added by apply_chat_template
clean_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

print(clean_text)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

**Movie Review: *The Dark Knight* (2008)**

**Title:** *The Dark Knight*  
**Director:** Christopher Nolan  
**Rating:** ★★★★★ (10/10)  

**Genre:** Action, Thriller, Crime  
**Runtime:** 140 minutes  

**A Classic Reimagined**  
Christopher Nolan’s *The Dark Knight* is a cinematic masterpiece that redefines the superhero genre. The film is a thrilling blend of action, suspense, and psychological depth, set against the backdrop of Gotham City in 2008. The story follows the Joker, a criminal mastermind who transforms into a terrifying figure known as the Joker, and we see him fight the Joker’s brother, Batman. The film is not just a superhero movie—it’s a psychological exploration of chaos, redemption, and the struggle between darkness and light.  

**The Joker: A Dark Knight of Chaos**  
The Joker’s character is one of the most unforgettable in cinematic history. His transformation into a criminal mastermind is both a physical and psychological transformation, and his ability to manipu